# This notebook is only for commands which I am not able to run from my local system



### 1-Chunking 
### 2-Loading into UC table

In [0]:
df=spark.read.csv("/Volumes/y_ws_250705/y_schema_for_ai/y_volume_for_rag/y_info_for_rag.txt")

In [0]:
df.show()

In [0]:
import pandas as pd
from pyspark.sql.functions import monotonically_increasing_id, col
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [0]:
import pandas as pd
from pyspark.sql.functions import monotonically_increasing_id, col
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- CONFIGURATION ---
CATALOG = "main"
SCHEMA = "default"
VOLUME_NAME = "my_volume"
FILE_NAME = "knowledge_base.txt"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.vector_search_source"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}/{FILE_NAME}"

# 1. Read and Chunk the Data
with open(VOLUME_PATH, "r") as f:
    text = f.read()

# Using Recursive splitter to maintain structural context
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_text(text)

# 2. Convert to Spark DataFrame
# Vector Search requires a Primary Key (id) and the text content
data = [{"id": i, "content": chunk} for i, chunk in enumerate(chunks)]
df = spark.createDataFrame(data)

# 3. Save as Delta Table with Change Data Feed (CDF)
# CDF allows the Vector Index to sync automatically when the table updates
(df.write.format("delta")
  .mode("overwrite")
  .option("delta.enableChangeDataFeed", "true")
  .saveAsTable(TABLE_NAME))

print(f"Success: Data indexed into Delta Table {TABLE_NAME}")